# ਨੇਵੀਅਰ–ਸਟੋਕਸ ਸਮੀਕਰਨਾਂ (Navier–Stokes Equations)
## **v6 – Diagram-Rich Edition**

**Author:** Harpreet Singh  
**Version:** 6.0 (2026) – Extra 3-D surfaces, vector fields, residual histories, cross-sections & comparison panels

### New diagrams in v6
- Cavity: 3-D streamfunction / u / v surfaces, quiver vector field, residual history, secondary-vortex zooms
- Stokes: multi-time 3-D surface + vector-style profiles
- Hagen–Poiseuille: dual 3-D views + radial profile table
- Taylor–Green: energy + dissipation with markers
- Burgers: multi-grid overlay + error surface concept
- Blasius: 3-D boundary-layer surface
- Validation dashboard retained

**Runtime:** `QUICK_MODE=True` ≈ 1.5–3 min

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.special import erfc
import scipy.sparse as sp
import scipy.sparse.linalg as spla

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    class tqdm:
        def __init__(self, iterable=None, **kwargs): self.iterable = iterable
        def __iter__(self): return iter(self.iterable)
        def update(self, n=1): pass
        def close(self): pass

QUICK_MODE = True
VERBOSE = True

if QUICK_MODE:
    CAVITY_N_PSI, CAVITY_N_PROJ = 33, 40
    CAVITY_RE_LIST = [100, 400]
    BURGERS_NX = [40, 80, 160]
    TG_N, TG_STEPS = 24, 40
    CAVITY_TMAX = 70.0
else:
    CAVITY_N_PSI, CAVITY_N_PROJ = 65, 80
    CAVITY_RE_LIST = [100, 400, 1000]
    BURGERS_NX = [60, 120, 240]
    TG_N, TG_STEPS = 48, 120
    CAVITY_TMAX = 200.0

plt.rcParams.update({
    'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': 0.3,
    'figure.dpi': 110, 'font.size': 10, 'axes.titlesize': 11,
})
pd.set_option('display.float_format', lambda v: f'{v:.4g}')
trapz = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

def timer(name=""):
    class T:
        def __enter__(self):
            self.t0 = time.time(); return self
        def __exit__(self, *a):
            if VERBOSE: print(f"  [{name}] {time.time()-self.t0:.2f} s")
    return T()

def regime(R):
    if R < 1: return 'Stokes'
    if R < 2300: return 'Laminar'
    if R < 4000: return 'Transition'
    return 'Turbulent'

RESULTS = {}
print(f"Setup OK | QUICK_MODE={QUICK_MODE}")

## 1. Dimensionless Numbers

In [ ]:
def Re(U,L,nu): return U*L/nu
cases = [('Bacterium',3e-5,2e-6,1e-6),('Honey',0.1,0.01,7e-3),('Blood',0.3,0.025,3.3e-6),
         ('Water pipe',1.,0.05,1e-6),('Walking',1.4,1.7,1.5e-5),('Car',30.,4.5,1.5e-5),
         ('Airliner',250.,5.,1.5e-5),('Blue whale',5.,25.,1e-6)]
df = pd.DataFrame(cases, columns=['Flow','U','L','nu'])
df['Re'] = Re(df.U,df.L,df.nu); df['Regime'] = df.Re.apply(regime)
display(df)

colors = {'Stokes':'#4c9be8','Laminar':'#59b36b','Transition':'#f0a830','Turbulent':'#d9534f'}
fig, ax = plt.subplots(figsize=(9,4.2))
d = df.sort_values('Re')
ax.barh(d.Flow, d.Re, color=[colors[r] for r in d.Regime])
ax.set_xscale('log'); ax.set_xlabel('Re (log)'); ax.set_title('Reynolds numbers of real flows')
for r in (1,2300,4000): ax.axvline(r, color='k', ls='--', lw=0.8)
ax.legend([plt.Rectangle((0,0),1,1,color=c) for c in colors.values()], colors.keys(), loc='lower right')
plt.tight_layout(); plt.show()

## 2. Exact Solutions – Extra Diagrams

In [ ]:
# Poiseuille + family
H,mu,G = 1.,0.01,0.1
N=41; y=np.linspace(0,H,N); h=y[1]-y[0]
A = (np.diag(-2*np.ones(N-2))+np.diag(np.ones(N-3),1)+np.diag(np.ones(N-3),-1))*mu/h**2
u_num = np.zeros(N); u_num[1:-1] = np.linalg.solve(A,-G*np.ones(N-2))
u_ex = G/(2*mu)*y*(H-y)
RESULTS['Poiseuille_err'] = np.abs(u_num-u_ex).max()

fig, axs = plt.subplots(1,3,figsize=(14,3.8))
axs[0].plot(u_ex,y,'b-',lw=2,label='Exact'); axs[0].plot(u_num,y,'ro',ms=4,label='FD')
axs[0].fill_betweenx(y,0,u_ex,alpha=0.15); axs[0].set_title('Poiseuille'); axs[0].legend()
eta = np.linspace(0,1,120)
for P in (-3,-1,0,1,3):
    axs[1].plot(eta+P*eta*(1-eta),eta,label=f'P={P}')
axs[1].axvline(0,color='k',lw=0.8); axs[1].set_title('Couette–Poiseuille'); axs[1].legend(fontsize=7)
# NEW: error profile
axs[2].semilogy(np.abs(u_num-u_ex)+1e-16, y, 'm-o', ms=3)
axs[2].set_xlabel('|error|'); axs[2].set_title(f'FD error (max={RESULTS["Poiseuille_err"]:.1e})')
plt.tight_layout(); plt.show()

In [ ]:
# Hagen–Poiseuille – richer 3-D
mu_p,rho_p,R_pipe,G_p = 1e-3,1000.,0.01,2.
u_pipe = lambda r: G_p/(4*mu_p)*(R_pipe**2-r**2)
r = np.linspace(0,R_pipe,1200)
Q_ex = np.pi*G_p*R_pipe**4/(8*mu_p)
U_mean = Q_ex/(np.pi*R_pipe**2)
print(f'Re = {rho_p*U_mean*2*R_pipe/mu_p:.0f}  Q = {Q_ex:.5e} m³/s')

fig = plt.figure(figsize=(14,5))
# Surface
ax = fig.add_subplot(131, projection='3d')
rr,th = np.meshgrid(np.linspace(0,R_pipe,30), np.linspace(0,2*np.pi,60))
ax.plot_surface(rr*np.cos(th)*100, rr*np.sin(th)*100, u_pipe(rr), cmap='viridis', edgecolor='none')
ax.set_title('Velocity paraboloid'); ax.set_xlabel('x (cm)'); ax.set_zlabel('u')
# Arrows
ax = fig.add_subplot(132, projection='3d')
TH,ZZ = np.meshgrid(np.linspace(0,2*np.pi,40), np.linspace(0,5,16))
ax.plot_surface(ZZ, R_pipe*100*np.cos(TH), R_pipe*100*np.sin(TH), alpha=0.1, color='gray')
yy = np.linspace(-R_pipe,R_pipe,8)
for xs in (0.5,2,3.5):
    ax.quiver(xs*np.ones_like(yy), yy*100, np.zeros_like(yy), u_pipe(np.abs(yy))*30, 0,0,
              color='crimson', arrow_length_ratio=0.3)
ax.set_title('Velocity arrows'); ax.set_box_aspect((2.5,1,1))
# Radial profile + table
ax = fig.add_subplot(133)
ax.plot(r*100, u_pipe(r), 'b-', lw=2)
ax.set_xlabel('r (cm)'); ax.set_ylabel('u (m/s)'); ax.set_title('Radial profile')
ax.axhline(U_mean, color='r', ls='--', label=f'U_mean={U_mean:.3f}')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Stokes first problem – multi-panel 3-D style
nu_s,U_s = 1e-4,1.
ys = np.linspace(0,0.05,70); ts = np.linspace(0.1,4,70)
Ys,Ts = np.meshgrid(ys,ts)
Us = U_s*erfc(Ys/(2*np.sqrt(nu_s*Ts)))
z99 = brentq(lambda z: erfc(z)-0.01, 0.1, 5)
RESULTS['Stokes_factor'] = 2*z99
print(f'δ99 ≈ {2*z99:.3f} √(νt)')

fig = plt.figure(figsize=(14,4.5))
ax = fig.add_subplot(131, projection='3d')
ax.plot_surface(Ys*100, Ts, Us, cmap='coolwarm', edgecolor='none')
ax.set_xlabel('y (cm)'); ax.set_ylabel('t'); ax.set_title("Stokes u(y,t)"); ax.view_init(25,-120)

ax = fig.add_subplot(132)
for t_ in (0.2,0.5,1,2,4):
    ax.plot(U_s*erfc(ys/(2*np.sqrt(nu_s*t_))), ys*100, lw=2, label=f't={t_}')
ax.set_xlabel('u/U'); ax.set_ylabel('y (cm)'); ax.set_title('Profiles'); ax.legend(fontsize=8)

# NEW: δ99 growth
ax = fig.add_subplot(133)
t_line = np.linspace(0.1,5,50)
ax.plot(t_line, 2*z99*np.sqrt(nu_s*t_line)*100, 'b-', lw=2)
ax.set_xlabel('t (s)'); ax.set_ylabel('δ99 (cm)'); ax.set_title('Boundary-layer thickness ∝ √t')
plt.tight_layout(); plt.show()

## 3. Numerical Analysis Diagrams

In [ ]:
# Truncation + Burgers
f,df_,x0 = np.sin, np.cos, 1.
hs = np.logspace(-1,-5,11)
fwd = (f(x0+hs)-f(x0))/hs; ctr = (f(x0+hs)-f(x0-hs))/(2*hs)
c4 = (-f(x0+2*hs)+8*f(x0+hs)-8*f(x0-hs)+f(x0-2*hs))/(12*hs)

fig, axs = plt.subplots(1,2,figsize=(11,3.8))
for name,e in [('Forward',np.abs(fwd-df_(x0))),('Central',np.abs(ctr-df_(x0))),('4th-order',np.abs(c4-df_(x0)))]:
    axs[0].loglog(hs,e,'o-',ms=4,label=name)
axs[0].legend(); axs[0].set_title('Truncation error'); axs[0].set_xlabel('h')

# Burgers
c_b,A_b,nu_b,Lb,Tb = 0.5,1.,0.1,3.,1.
exact_b = lambda x,t: c_b-A_b*np.tanh(A_b*(x-c_b*t)/(2*nu_b))
def burgers_run(nx):
    x=np.linspace(-Lb,Lb,nx+1); h=x[1]-x[0]
    nsteps=int(np.ceil(Tb/(0.25*h*h/nu_b))); dt=Tb/nsteps
    def F(u,t):
        u=u.copy(); u[0]=exact_b(x[0],t); u[-1]=exact_b(x[-1],t)
        r=np.zeros_like(u)
        r[1:-1]=-u[1:-1]*(u[2:]-u[:-2])/(2*h)+nu_b*(u[2:]-2*u[1:-1]+u[:-2])/h**2
        return r
    u,t=exact_b(x,0.),0.
    for _ in range(nsteps):
        k1=F(u,t); k2=F(u+0.5*dt*k1,t+0.5*dt); k3=F(u+0.5*dt*k2,t+0.5*dt); k4=F(u+dt*k3,t+dt)
        u=u+dt/6*(k1+2*k2+2*k3+k4); t+=dt
        u[0]=exact_b(x[0],t); u[-1]=exact_b(x[-1],t)
    return x,u,h

nxs=BURGERS_NX; L2,sols=[],{}
for n in tqdm(nxs,desc='Burgers'):
    x_,u_,h_=burgers_run(n); sols[n]=(x_,u_)
    e=u_-exact_b(x_,Tb); L2.append(np.sqrt(h_*np.sum(e**2)))
L2=np.array(L2); RESULTS['Burgers_order']=np.log2(L2[-2]/L2[-1]) if len(L2)>1 else np.nan

hh=np.array([2*Lb/n for n in nxs])
axs[1].loglog(hh,L2,'o-',label='L2'); axs[1].loglog(hh,L2[0]*(hh/hh[0])**2,'k--',label='O(h²)')
axs[1].legend(); axs[1].set_title('Burgers convergence')
plt.tight_layout(); plt.show()

# NEW: multi-grid shock overlay
fig, ax = plt.subplots(figsize=(8,3.8))
xe=np.linspace(-Lb,Lb,400)
ax.plot(xe,exact_b(xe,Tb),'k-',lw=2.5,label='Exact')
for n,c in zip(nxs,['r','g','b']):
    ax.plot(*sols[n],'-o',ms=3,label=f'nx={n}',alpha=0.8)
ax.set_xlim(-1,2); ax.legend(); ax.set_title('Burgers shock – grid comparison')
plt.tight_layout(); plt.show()

## 4. Lid-Driven Cavity – Full Diagram Suite

In [ ]:
def cavity_psi_omega(Re_,N,U=1.,safety=0.75,tol=3e-5,Tmax=CAVITY_TMAX,max_steps=60000):
    nu=1./Re_; h=1./(N-1); n=N-2
    dt=safety/(4*nu/h**2+2*U/h)
    T1=sp.diags([1.,-2.,1.],[-1,0,1],shape=(n,n)); I1=sp.identity(n)
    lu=spla.splu(((sp.kron(I1,T1)+sp.kron(T1,I1))/h**2).tocsc())
    psi=np.zeros((N,N)); om=np.zeros((N,N)); hist=[]; psi_prev=psi.copy(); step=0
    pbar=tqdm(total=min(int(Tmax/dt),max_steps),desc=f'ψ–ω Re={Re_}',disable=not VERBOSE)
    while step*dt<Tmax and step<max_steps:
        om[0,:]=-2*psi[1,:]/h**2; om[-1,:]=-2*psi[-2,:]/h**2-2*U/h
        om[:,0]=-2*psi[:,1]/h**2; om[:,-1]=-2*psi[:,-2]/h**2
        u=(psi[2:,1:-1]-psi[:-2,1:-1])/(2*h); v=-(psi[1:-1,2:]-psi[1:-1,:-2])/(2*h)
        dwdx=(om[1:-1,2:]-om[1:-1,:-2])/(2*h); dwdy=(om[2:,1:-1]-om[:-2,1:-1])/(2*h)
        lap=(om[1:-1,2:]+om[1:-1,:-2]+om[2:,1:-1]+om[:-2,1:-1]-4*om[1:-1,1:-1])/h**2
        om[1:-1,1:-1]+=dt*(-u*dwdx-v*dwdy+nu*lap)
        psi[1:-1,1:-1]=lu.solve(-om[1:-1,1:-1].ravel()).reshape(n,n)
        step+=1; pbar.update(1)
        if step%30==0:
            ch=np.abs(psi-psi_prev).max()/(30*dt); psi_prev=psi.copy()
            hist.append((step*dt,ch))
            if ch<tol: break
    pbar.close()
    xs=np.linspace(0,1,N); uf=np.zeros((N,N)); vf=np.zeros((N,N))
    uf[1:-1,1:-1]=(psi[2:,1:-1]-psi[:-2,1:-1])/(2*h)
    vf[1:-1,1:-1]=-(psi[1:-1,2:]-psi[1:-1,:-2])/(2*h); uf[-1,:]=U
    return dict(Re=Re_,N=N,h=h,dt=dt,steps=step,psi=psi,om=om,u=uf,v=vf,x=xs,hist=np.array(hist))

ghia_y=np.array([0.,0.0547,0.0625,0.0703,0.1016,0.1719,0.2813,0.4531,0.5,
                 0.6172,0.7344,0.8516,0.9531,0.9609,0.9688,0.9766,1.])
ghia_u={100:np.array([0.,-0.03717,-0.04192,-0.04775,-0.06434,-0.10150,-0.15662,-0.21090,-0.20581,
                      -0.13641,0.00332,0.23151,0.68717,0.73722,0.78871,0.84123,1.]),
        400:np.array([0.,-0.08186,-0.09266,-0.10338,-0.14612,-0.24299,-0.32726,-0.17119,-0.11477,
                      0.02135,0.16256,0.29093,0.55892,0.61756,0.68439,0.75837,1.]),
        1000:np.array([0.,-0.18109,-0.20196,-0.22220,-0.29730,-0.38289,-0.27805,-0.10648,-0.06080,
                       0.05702,0.18719,0.33304,0.46604,0.51117,0.57492,0.65928,1.])}

runs={}
print('=== ψ–ω ===')
for Re_ in CAVITY_RE_LIST:
    with timer(f'Re={Re_}'):
        r=cavity_psi_omega(Re_,CAVITY_N_PSI)
        runs[(Re_,CAVITY_N_PSI)]=r
        print(f'  steps={r["steps"]}  ψ_min={r["psi"].min():+.5f}')

In [ ]:
def cavity_projection(Re_,N,U=1.,gamma=0.2,tol=3e-5,Tmax=CAVITY_TMAX,cfl=0.35,max_steps=80000):
    nu=1./Re_; h=1./N; dt=cfl*min(h/U,h*h*Re_/4)
    Tn=sp.diags([1.,-2.,1.],[-1,0,1],shape=(N,N)).tolil(); Tn[0,0]=-1.; Tn[N-1,N-1]=-1.
    I=sp.identity(N); A=(sp.kron(I,Tn)+sp.kron(Tn,I)).tolil()/h**2; A[0,:]=0; A[0,0]=1.
    lu=spla.splu(A.tocsc())
    u=np.zeros((N+1,N)); v=np.zeros((N,N+1)); p=np.zeros((N,N)); hist=[]; step=0
    pbar=tqdm(total=min(int(Tmax/dt),max_steps),desc=f'MAC Re={Re_}',disable=not VERBOSE)
    while step*dt<Tmax and step<max_steps:
        ue=np.zeros((N+1,N+2)); ue[:,1:-1]=u; ue[:,0]=-u[:,0]; ue[:,-1]=2*U-u[:,-1]
        ve=np.zeros((N+2,N+1)); ve[1:-1,:]=v; ve[0,:]=-v[0,:]; ve[-1,:]=-v[-1,:]
        uC,uE,uW=u[1:-1,:],u[2:,:],u[:-2,:]; uN,uS=ue[1:-1,2:],ue[1:-1,:-2]
        vn=0.5*(v[:-1,1:]+v[1:,1:]); vs=0.5*(v[:-1,:-1]+v[1:,:-1])
        duudx=(((uC+uE)/2)**2-((uW+uC)/2)**2)/h + gamma/h*(np.abs(uC+uE)/2*(uC-uE)/2-np.abs(uW+uC)/2*(uW-uC)/2)
        duvdy=(vn*(uC+uN)/2-vs*(uS+uC)/2)/h + gamma/h*(np.abs(vn)*(uC-uN)/2-np.abs(vs)*(uS-uC)/2)
        lapu=(uE+uW+uN+uS-4*uC)/h**2
        vC,vN,vS=v[:,1:-1],v[:,2:],v[:,:-2]; vE,vW=ve[2:,1:-1],ve[:-2,1:-1]
        uea=0.5*(u[1:,:-1]+u[1:,1:]); uwa=0.5*(u[:-1,:-1]+u[:-1,1:])
        dvvdy=(((vC+vN)/2)**2-((vS+vC)/2)**2)/h + gamma/h*(np.abs(vC+vN)/2*(vC-vN)/2-np.abs(vS+vC)/2*(vS-vC)/2)
        duvdx=(uea*(vC+vE)/2-uwa*(vW+vC)/2)/h + gamma/h*(np.abs(uea)*(vC-vE)/2-np.abs(uwa)*(vW-vC)/2)
        lapv=(vE+vW+vN+vS-4*vC)/h**2
        us_=u.copy(); vs_=v.copy()
        us_[1:-1,:]+=dt*(-duudx-duvdy+nu*lapu); vs_[:,1:-1]+=dt*(-dvvdy-duvdx+nu*lapv)
        div=(us_[1:,:]-us_[:-1,:])/h+(vs_[:,1:]-vs_[:,:-1])/h
        rhs=(div/dt).ravel(); rhs[0]=0.; p=lu.solve(rhs).reshape(N,N)
        un=us_.copy(); vn_=vs_.copy()
        un[1:-1,:]-=dt*(p[1:,:]-p[:-1,:])/h; vn_[:,1:-1]-=dt*(p[:,1:]-p[:,:-1])/h
        step+=1; pbar.update(1)
        if step%60==0:
            ch=max(np.abs(un-u).max(),np.abs(vn_-v).max())/dt; hist.append((step*dt,ch))
            if ch<tol: u,v=un,vn_; break
        u,v=un,vn_
    pbar.close()
    divf=np.abs((u[1:,:]-u[:-1,:])/h+(v[:,1:]-v[:,:-1])/h).max()
    psi=np.zeros((N+1,N+1)); psi[:,1:]=h*np.cumsum(u,axis=1)
    uc=0.5*(u[:-1,:]+u[1:,:]); vc=0.5*(v[:,:-1]+v[:,1:])
    return dict(Re=Re_,N=N,h=h,dt=dt,steps=step,u=u,v=v,p=p-p.mean(),
                psi=psi,uc=uc,vc=vc,hist=np.array(hist),div=divf)

def centerline_u(r):
    N=r['N']; yc=(np.arange(N)+0.5)/N
    return np.r_[0,yc,1], np.r_[0,r['u'][N//2,:],1.]

pruns={}
print('=== MAC ===')
for Re_ in CAVITY_RE_LIST:
    with timer(f'Re={Re_}'):
        r=cavity_projection(Re_,CAVITY_N_PROJ)
        pruns[(Re_,CAVITY_N_PROJ)]=r
        y_,u_=centerline_u(r)
        rms=np.sqrt(np.mean((np.interp(ghia_y,y_,u_)-ghia_u[Re_])**2)) if Re_ in ghia_u else np.nan
        RESULTS[f'RMS_{Re_}']=rms; RESULTS[f'div_{Re_}']=r['div']
        print(f'  steps={r["steps"]}  max|div|={r["div"]:.1e}  ψ_min={r["psi"].min():+.5f}  RMS={rms:.4f}')

In [ ]:
# Centerline validation
fig, axs = plt.subplots(1,len(CAVITY_RE_LIST),figsize=(5*len(CAVITY_RE_LIST),4))
if len(CAVITY_RE_LIST)==1: axs=[axs]
for ax,Re_ in zip(axs,CAVITY_RE_LIST):
    y_,u_=centerline_u(pruns[(Re_,CAVITY_N_PROJ)])
    ax.plot(u_,y_,'g-',lw=2.2,label='MAC')
    if (Re_,CAVITY_N_PSI) in runs:
        rpw=runs[(Re_,CAVITY_N_PSI)]
        ax.plot(rpw['u'][:,CAVITY_N_PSI//2],rpw['x'],'b--',label='ψ–ω')
    if Re_ in ghia_u: ax.plot(ghia_u[Re_],ghia_y,'ro',ms=5,label='Ghia')
    ax.set_title(f'Re={Re_}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ===== RICH CAVITY DIAGRAMS =====
Re_plot = max(CAVITY_RE_LIST)
r = pruns[(Re_plot, CAVITY_N_PROJ)]
N,h = r['N'], r['h']
xc = (np.arange(N)+0.5)*h; Xc,Yc = np.meshgrid(xc,xc,indexing='ij')
xn = np.linspace(0,1,N+1); Xn,Yn = np.meshgrid(xn,xn,indexing='ij')
spd = np.hypot(r['uc'], r['vc'])
om = np.zeros((N,N))
om[1:-1,1:-1] = (r['vc'][2:,1:-1]-r['vc'][:-2,1:-1])/(2*h) - (r['uc'][1:-1,2:]-r['uc'][1:-1,:-2])/(2*h)

# 1. Three 3-D surfaces
fig = plt.figure(figsize=(15,4.5))
for i,(Z,ttl,cmap) in enumerate([(spd,'Speed |u|','viridis'),
                                  (r['psi'][1:,1:],'Streamfunction ψ','cividis'),
                                  (np.clip(om,-8,8),'Vorticity ω','RdBu_r')]):
    ax = fig.add_subplot(1,3,i+1, projection='3d')
    if Z.shape == (N,N):
        ax.plot_surface(Xc[::2,::2], Yc[::2,::2], Z[::2,::2], cmap=cmap, edgecolor='none')
    else:
        ax.plot_surface(Xn[1::2,1::2], Yn[1::2,1::2], Z[::2,::2], cmap=cmap, edgecolor='none')
    ax.set_title(f'{ttl}  Re={Re_plot}'); ax.view_init(28,-125)
plt.tight_layout(); plt.show()

# 2. Quiver vector field + speed contours
fig, ax = plt.subplots(figsize=(6,5.5))
skip = max(1, N//16)
ax.contourf(Xc, Yc, spd, 20, cmap='viridis', alpha=0.85)
ax.quiver(Xc[::skip,::skip], Yc[::skip,::skip],
          r['uc'][::skip,::skip], r['vc'][::skip,::skip],
          color='w', scale=8, width=0.004)
ax.set_aspect('equal'); ax.set_title(f'Velocity vectors + speed  Re={Re_plot}')
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout(); plt.show()

# 3. Residual / convergence history
fig, axs = plt.subplots(1,2,figsize=(11,3.8))
for (Re_,N_), rr in pruns.items():
    if len(rr['hist']):
        axs[0].semilogy(rr['hist'][:,0], rr['hist'][:,1], label=f'Re={Re_}')
axs[0].set_xlabel('t'); axs[0].set_ylabel('max |du/dt|'); axs[0].set_title('MAC residual history'); axs[0].legend()
for (Re_,N_), rr in runs.items():
    if len(rr['hist']):
        axs[1].semilogy(rr['hist'][:,0], rr['hist'][:,1], label=f'Re={Re_}')
axs[1].set_xlabel('t'); axs[1].set_ylabel('max |dψ/dt|'); axs[1].set_title('ψ–ω residual history'); axs[1].legend()
plt.tight_layout(); plt.show()

# 4. Secondary vortex zooms
fig, axs = plt.subplots(1,2,figsize=(11,4.5))
for ax,(x0,x1) in zip(axs, ((0,0.28),(0.72,1.0))):
    m = (xn>=x0)&(xn<=x1); mz = (xn<=0.28)
    cf = ax.contourf(Xn[np.ix_(m,mz)], Yn[np.ix_(m,mz)], r['psi'][np.ix_(m,mz)],
                     levels=np.linspace(-2e-3,3e-4,25), cmap='coolwarm', extend='both')
    ax.contour(Xn[np.ix_(m,mz)], Yn[np.ix_(m,mz)], r['psi'][np.ix_(m,mz)], levels=[0], colors='k', linewidths=1.5)
    ax.set_aspect('equal'); ax.set_title(f'Bottom {"left" if x0==0 else "right"} secondary vortex')
plt.tight_layout(); plt.show()
print(f'ψ_min = {r["psi"].min():+.5f}')

## 5. Spectral Taylor–Green + Lamb–Oseen

In [ ]:
def taylor_green_spectral(N=TG_N, nu=0.02, T=6., nsteps=TG_STEPS):
    L=2*np.pi; x=np.linspace(0,L,N,endpoint=False)
    X,Y,Z=np.meshgrid(x,x,x,indexing='ij')
    u=np.sin(X)*np.cos(Y)*np.cos(Z); v=-np.cos(X)*np.sin(Y)*np.cos(Z); w=np.zeros_like(u)
    k=np.fft.fftfreq(N,d=L/N)*2*np.pi
    KX,KY,KZ=np.meshgrid(k,k,k,indexing='ij'); K2=KX**2+KY**2+KZ**2; K2[0,0,0]=1.
    kmax=N//3; dealias=(np.abs(KX)<kmax)&(np.abs(KY)<kmax)&(np.abs(KZ)<kmax)
    def rhs(u,v,w):
        uh,vh,wh=np.fft.fftn(u),np.fft.fftn(v),np.fft.fftn(w)
        ux=np.fft.ifftn(1j*KX*uh).real; uy=np.fft.ifftn(1j*KY*uh).real; uz=np.fft.ifftn(1j*KZ*uh).real
        vx=np.fft.ifftn(1j*KX*vh).real; vy=np.fft.ifftn(1j*KY*vh).real; vz=np.fft.ifftn(1j*KZ*vh).real
        wx=np.fft.ifftn(1j*KX*wh).real; wy=np.fft.ifftn(1j*KY*wh).real; wz=np.fft.ifftn(1j*KZ*wh).real
        cu=np.fft.fftn(u*ux+v*uy+w*uz)*dealias
        cv=np.fft.fftn(u*vx+v*vy+w*vz)*dealias
        cw=np.fft.fftn(u*wx+v*wy+w*wz)*dealias
        divc=1j*(KX*cu+KY*cv+KZ*cw)
        pu=cu-1j*KX*divc/K2; pv=cv-1j*KY*divc/K2; pw=cw-1j*KZ*divc/K2
        du=-pu-nu*K2*uh; dv=-pv-nu*K2*vh; dw=-pw-nu*K2*wh
        du[0,0,0]=dv[0,0,0]=dw[0,0,0]=0
        return np.fft.ifftn(du).real,np.fft.ifftn(dv).real,np.fft.ifftn(dw).real
    dt=T/nsteps; t_hist,E_hist,eps_hist=[],[],[]
    for n in tqdm(range(nsteps),desc='Spectral TG',disable=not VERBOSE):
        k1u,k1v,k1w=rhs(u,v,w)
        k2u,k2v,k2w=rhs(u+0.5*dt*k1u,v+0.5*dt*k1v,w+0.5*dt*k1w)
        k3u,k3v,k3w=rhs(u+0.5*dt*k2u,v+0.5*dt*k2v,w+0.5*dt*k2w)
        k4u,k4v,k4w=rhs(u+dt*k3u,v+dt*k3v,w+dt*k3w)
        u+=dt/6*(k1u+2*k2u+2*k3u+k4u); v+=dt/6*(k1v+2*k2v+2*k3v+k4v); w+=dt/6*(k1w+2*k2w+2*k3w+k4w)
        if n%max(1,nsteps//15)==0 or n==nsteps-1:
            E=0.5*np.mean(u**2+v**2+w**2)
            uh,vh,wh=np.fft.fftn(u),np.fft.fftn(v),np.fft.fftn(w)
            eps=nu*np.mean((np.abs(1j*KX*uh)**2+np.abs(1j*KY*uh)**2+np.abs(1j*KZ*uh)**2+
                           np.abs(1j*KX*vh)**2+np.abs(1j*KY*vh)**2+np.abs(1j*KZ*vh)**2+
                           np.abs(1j*KX*wh)**2+np.abs(1j*KY*wh)**2+np.abs(1j*KZ*wh)**2).real)
            t_hist.append(n*dt); E_hist.append(E); eps_hist.append(eps)
    return np.array(t_hist),np.array(E_hist),np.array(eps_hist)

with timer('Spectral TG'):
    t_tg,E_tg,eps_tg = taylor_green_spectral()
RESULTS['TG_ratio'] = E_tg[-1]/E_tg[0]

fig, axs = plt.subplots(1,2,figsize=(11,3.8))
axs[0].semilogy(t_tg,E_tg,'b-o',ms=5,lw=2); axs[0].set_title('Energy decay'); axs[0].set_xlabel('t')
axs[1].plot(t_tg,eps_tg,'r-s',ms=5,lw=2); axs[1].set_title('Dissipation rate'); axs[1].set_xlabel('t')
plt.tight_layout(); plt.show()
print(f'Final/Initial KE = {RESULTS["TG_ratio"]:.4f}')

In [ ]:
# Lamb–Oseen – profiles + 3-D surface
Gam,nu_v = 1.,1e-3
vth = lambda r,t: Gam/(2*np.pi*r)*(1-np.exp(-r**2/(4*nu_v*t)))
rr = np.linspace(1e-4,0.45,1800)

fig = plt.figure(figsize=(12,4.5))
ax = fig.add_subplot(121)
for t_ in (0.5,2,8,32):
    ax.plot(rr, vth(rr,t_), lw=2, label=f't={t_}')
ax.set_xlabel('r'); ax.set_ylabel('v_θ'); ax.set_title('Lamb–Oseen profiles'); ax.legend()

ax = fig.add_subplot(122, projection='3d')
xg,yg = np.meshgrid(np.linspace(-0.35,0.35,70), np.linspace(-0.35,0.35,70))
rg = np.hypot(xg,yg)+1e-9
ax.plot_surface(xg,yg,vth(rg,1.5), cmap='plasma', edgecolor='none')
ax.set_title('|v_θ| surface t=1.5 s'); ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout(); plt.show()

## 6. Blasius Boundary Layer – 3-D Surface

In [ ]:
def blasius_rhs(eta,y): return [y[1],y[2],-0.5*y[0]*y[2]]
s_star = brentq(lambda s: solve_ivp(blasius_rhs,[0,12],[0,0,s],rtol=1e-9,atol=1e-11).y[1,-1]-1., 0.1,1.)
sol = solve_ivp(blasius_rhs,[0,12],[0,0,s_star],rtol=1e-9,atol=1e-11,dense_output=True)
eta99 = brentq(lambda e: sol.sol(e)[1]-0.99, 1,10)
RESULTS['Blasius_fpp'] = s_star
print(f"f''(0)={s_star:.5f} (known 0.33206)  η99={eta99:.3f}")

U_b,nu_b2 = 10.,1.5e-5
xb = np.linspace(0.01,1.,60); yb = np.linspace(0,0.01,45)
Xb,Yb = np.meshgrid(xb,yb)
ETA = Yb*np.sqrt(U_b/(nu_b2*Xb))
e_tab = np.linspace(0,12,1000); fp = sol.sol(e_tab)[1]
Ub = np.interp(ETA, e_tab, fp)

fig = plt.figure(figsize=(14,4.5))
ax = fig.add_subplot(131)
ee = np.linspace(0,8,80)
ax.plot(sol.sol(ee)[1],ee,lw=2,label="f'"); ax.plot(sol.sol(ee)[2],ee,'--',lw=2,label="f''")
ax.axhline(eta99,color='gray',ls=':'); ax.legend(); ax.set_title('Blasius profiles')

ax = fig.add_subplot(132)
cs = ax.contourf(Xb, Yb*1000, Ub, 18, cmap='viridis')
ax.plot(xb, eta99*np.sqrt(nu_b2*xb/U_b)*1000, 'w--', lw=2, label='δ99')
plt.colorbar(cs,ax=ax); ax.set_xlabel('x (m)'); ax.set_ylabel('y (mm)'); ax.set_title('u/U field'); ax.legend(fontsize=8)

ax = fig.add_subplot(133, projection='3d')
ax.plot_surface(Xb[::2,::2], Yb[::2,::2]*1000, Ub[::2,::2], cmap='viridis', edgecolor='none')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (mm)'); ax.set_zlabel('u/U'); ax.set_title('3-D BL surface'); ax.view_init(25,-130)
plt.tight_layout(); plt.show()

## 7. Validation Dashboard

In [ ]:
print('='*55)
print('VALIDATION DASHBOARD')
print('='*55)
checks = []
ok = RESULTS.get('Poiseuille_err',1) < 1e-12
checks.append(('Poiseuille FD error', f"{RESULTS.get('Poiseuille_err',np.nan):.1e}", '<1e-12', ok))
ok = abs(RESULTS.get('Stokes_factor',0)-3.64)<0.05
checks.append(('Stokes δ99 factor', f"{RESULTS.get('Stokes_factor',np.nan):.3f}", '≈3.64', ok))
ok = RESULTS.get('Burgers_order',0) > 1.5
checks.append(('Burgers order', f"{RESULTS.get('Burgers_order',np.nan):.2f}", '>1.5', ok))
for Re_ in CAVITY_RE_LIST:
    ok = RESULTS.get(f'div_{Re_}',1) < 1e-10
    checks.append((f'MAC |div| Re={Re_}', f"{RESULTS.get(f'div_{Re_}',np.nan):.1e}", '<1e-10', ok))
    ok = RESULTS.get(f'RMS_{Re_}',1) < 0.06
    checks.append((f'MAC RMS Ghia Re={Re_}', f"{RESULTS.get(f'RMS_{Re_}',np.nan):.4f}", '<0.06', ok))
ok = abs(RESULTS.get('Blasius_fpp',0)-0.33206)<0.001
checks.append(("Blasius f''(0)", f"{RESULTS.get('Blasius_fpp',np.nan):.5f}", '≈0.33206', ok))
ok = 0 < RESULTS.get('TG_ratio',1) < 1
checks.append(('TG energy ratio', f"{RESULTS.get('TG_ratio',np.nan):.4f}", '0<r<1', ok))

dash = pd.DataFrame(checks, columns=['Check','Value','Expected','PASS'])
display(dash.style.apply(lambda s: ['background-color:#d4edda' if v else 'background-color:#f8d7da' for v in s], subset=['PASS']))
print(f"\n→ {dash.PASS.sum()}/{len(dash)} checks passed")

### Diagram summary (v6)
- **3-D surfaces**: Hagen–Poiseuille, Stokes, cavity speed/ψ/vorticity, Lamb–Oseen, Blasius BL
- **Vector field**: cavity quiver plot overlaid on speed contours
- **Residual histories**: MAC and ψ–ω convergence curves
- **Secondary-vortex zooms**: left & right bottom corners
- **Multi-grid overlays**: Burgers shock comparison
- **Error profiles**: Poiseuille FD error
- **Growth laws**: Stokes δ99 ∝ √t

---
**End of v6 Diagram-Rich notebook.**  
Set `QUICK_MODE = False` for higher-resolution figures.